# Ionization potential of Cl⁻ in NaCl (PySCF on eDFTpy embedding potentials)

This notebook is the last step of the *Ionization Potential* tutorial. It uses the embedding potentials (EP) written by the two eDFTpy runs:

| file | run | used for |
|---|---|---|
| `standard/sub_cl_0.snpy` | `standard/` | neutral EP (static) |
| `oxidized/sub_cl_c.snpy` | `oxidized/` | impurity EP (dynamic) |

and computes, for Cl⁻ → Cl + e⁻,

$$\mathrm{IP} = E[\mathrm{Cl}] - E[\mathrm{Cl}^-]$$

in vacuum (no EP), with the neutral EP, and with the impurity-model EP. It needs PySCF with embedding-potential support (`vemb_mat`) and `common.py` in the same folder.

In [ ]:
import os
import numpy as np
from common import run_state

Ha2eV = 27.211386245988

method = 'pbe'          # 'pbe', 'mp2' or 'ccsd'

geom_file    = os.path.abspath('standard/sub_cl_0.xyz')
neutral_ep   = os.path.abspath('standard/sub_cl_0.snpy')    # V_emb static
impurity_ep  = os.path.abspath('oxidized/sub_cl_c.snpy')    # V_emb dynamic

# shift S = 1/N Tr[D (V_static - V_dynamic)]
shift_spec = dict(static_extemb=neutral_ep, dynamic_extemb=impurity_ep)

## Run the three states

Each `run_state` does an isolated pass (`e_iso`, no EP) and an embedded pass (`e_embd`, with the EP as a one-electron potential), and returns the interaction energy `e_fde` = Tr[D V_emb] and the shift S.

| state | charge | 2S | EP |
|---|---|---|---|
| Cl⁻ | −1 | 0 | neutral |
| Cl  | 0 | 1 | neutral |
| Cl  | 0 | 1 | impurity |

In [ ]:
cl_anion   = run_state(method, geom_file, spin=0, charge=-1, extemb=neutral_ep,  shift_spec=shift_spec)
cl_neutral = run_state(method, geom_file, spin=1, charge=0,  extemb=neutral_ep,  shift_spec=shift_spec)
cl_impur   = run_state(method, geom_file, spin=1, charge=0,  extemb=impurity_ep, shift_spec=shift_spec)

for name, st in (('Cl-  (neutral EP)', cl_anion), ('Cl   (neutral EP)', cl_neutral), ('Cl   (impurity EP)', cl_impur)):
    print(f"{name:20s} E_iso = {st['e_iso']:.6f}  E_embd = {st['e_embd']:.6f}  E_fde = {st['e_fde']:.6f}  S = {st['shift']:.6f} Ha")

## Ionization potential

In [ ]:
def ip(final, initial, key):
    return (final[key] - initial[key]) * Ha2eV

def ip_fde(final, initial):
    return ((final['e_embd'] + final['e_fde']) - (initial['e_embd'] + initial['e_fde'])) * Ha2eV

print(f"Method: {method.upper()}")
print(f"IP vacuum (no EP)                      : {ip(cl_neutral, cl_anion, 'e_iso'):8.4f} eV   (exp. EA of Cl = 3.61 eV)")
print(f"IP neutral EP   (E_embd)               : {ip(cl_neutral, cl_anion, 'e_embd'):8.4f} eV")
print(f"IP neutral EP   (E_embd + E_fde)       : {ip_fde(cl_neutral, cl_anion):8.4f} eV")
print(f"IP impurity EP  (E_embd)               : {ip(cl_impur, cl_anion, 'e_embd'):8.4f} eV")
print(f"IP impurity EP  (E_embd + E_fde)       : {ip_fde(cl_impur, cl_anion):8.4f} eV")
print(f"Shift S (Cl, impurity EP)              : {cl_impur['shift'] * Ha2eV:8.4f} eV")

## Potential vs SHE (optional)

Oxidation potential on the SHE scale, with the absolute SHE potential 4.44 V (Trasatti 1986).

In [ ]:
SHE_abs = 4.44  # V
print(f"E_oxidation vs SHE (impurity EP, E_embd + E_fde): {ip_fde(cl_impur, cl_anion) - SHE_abs:.4f} V")